# Appendix B: Contractions and Other Inner Products

Printed source span verified before authoring: Appendix B is printed pages 589-596, represented by PDF pages 605-612 in the text layer. This notebook is an original, executable reference for the convention problem that Appendix B addresses. Geometric algebra literature uses several products that look like inner products but disagree in exactly the cases where implementation details matter: scalar arguments, grade order, and degenerate inputs.

The purpose here is not to pick a universal notation for every reader. It is to make the conventions explicit enough that code can test them. The book favors left contraction because its asymmetry carries geometric information: `A left-contract B` is naturally zero when the first argument has too high a grade to be contained in the second. A dot-style product can be more symmetric, but it has to branch on grade order. The Hestenes inner product adds another scalar special case. All three can agree in common lower-grade-first situations, but they should not be treated as interchangeable in an implementation.


## Translation Guide

The key translation is from notation to function behavior. Left contraction lowers the grade of the second argument by the grade of the first. Right contraction lowers the grade of the first argument by the grade of the second. The Appendix B dot product chooses whichever side has the lower grade. The Hestenes inner product follows that dot product except that it returns zero if either input is scalar.

The conventions have different failure modes. A left contraction that returns zero because the first grade is too large is reporting a useful containment failure. A symmetric dot product hides that failure by switching to the other contraction. That can be pleasant in hand calculations and inconvenient in algorithms that interpret grades dynamically. Scalar handling is another fault line: if scalar multiplication is supposed to pass through a product, Hestenes-style zero scalar cases need explicit care. The implementation route is to table the products, test the grade outputs, and keep convention names in artifacts. When a formula is copied from another source, first ask which inner product its dot means.


## Route

This notebook follows the appendix from convention to proof technique. First, we record the page-span audit. Then we tabulate product variants on representative blades. Next, we isolate cases where left contraction, right contraction, dot product, and Hestenes inner product agree or diverge. We use a recursive vector-on-blade contraction check to mirror the proof technique without reproducing the proof. Finally, we test projection and degenerate behavior, because those are the places where a wrong convention tends to produce plausible but incorrect numerical output.


In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
BOOK_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils" / "ga").exists(): BOOK_ROOT = candidate; break
if str(BOOK_ROOT) not in sys.path: sys.path.insert(0, str(BOOK_ROOT))
from utils.ga import Algebra
from utils.appendix_reference import (
    APPENDIX_ARTIFACT_ROOT, basis_blades, blade_square, dot_product, hestenes_inner,
    inner_scalar, metric_signature, operation_records, projection_onto_blade,
    rejection_from_blade, source_span_records, vector_contract_blade_recursive,
    write_csv_artifact, write_json_artifact,
)
pd.set_option("display.max_columns", 14)
APPENDIX_KEY = "appendix-b"
artifact_paths = {}
def remember(label, path): artifact_paths[label] = str(Path(path)); return Path(path)
print("project", BOOK_ROOT)


project D:\Geometry\Geometric-Algebra-for-Computer-Science


In [2]:
source_rows = source_span_records()
source_payload = {"pdf":"Geometric Algebra for Computer Science.pdf","verification_tools":["pdfinfo","pdftotext -layout"],"pdf_page_count":640,"note":"Printed pages are authoritative; omitted blank pages are listed explicitly.","records":source_rows}
remember("source_span_json", write_json_artifact(["source"], "source-span-audit.json", source_payload))
remember("source_span_csv", write_csv_artifact(["source"], "source-span-audit.csv", source_rows))
pd.DataFrame(source_rows)


,appendix,title,printed_span,text_pdf_pages,missing_or_blank_printed_pages,sections
0,A,Metrics and Null Vectors,585-588,"[602, 603, 604]",[588],"[A.1 bilinear form, A.2 diagonalization to ort..."
1,B,Contractions and Other Inner Products,589-596,"[605, 606, 607, 608, 609, 610, 611, 612]",[],"[B.1 other inner products, B.2 equivalence of ..."
2,C,Subspace Products Retrieved,597-602,"[613, 614, 615, 616, 617]",[602],"[C.1 outer product from geometric product, C.2..."
3,D,Common Equations,603-608,"[618, 619, 620, 621, 622]",[608],"[D.1 product and operation notation, D.2 sign ..."


## Product Convention Table

A convention table is a better reference than a paragraph when debugging. The rows below include scalars, vectors, bivectors, and a trivector. The table is small enough to inspect and large enough to reveal the branches. Watch the `left_contract`, `right_contract`, `dot_product`, and `hestenes_inner` columns. Equal-grade nonscalar cases usually agree as scalar products. Lower-grade-first cases also agree. Scalar and higher-grade-first cases are where the definitions announce themselves.


In [3]:
alg = Algebra([1, 1, 1], names=["e1", "e2", "e3"])
e1, e2, e3 = alg.basis()
e12 = e1.wedge(e2); e23 = e2.wedge(e3); I = alg.pseudoscalar()
left_items = {"1": alg.scalar(2), "e1": e1, "e12": e12, "I": I}
right_items = {"1": alg.scalar(3), "e2": e2, "e12": e12, "e23": e23, "I": I}
convention_rows = operation_records(alg, left_items, right_items)
remember("convention_table_csv", write_csv_artifact([APPENDIX_KEY,"tables"], "inner-product-conventions.csv", convention_rows))
pd.DataFrame(convention_rows)


,left,right,left_grade,right_grade,gp,wedge,left_contract,right_contract,dot_product,hestenes_inner
0,1,1,0,0,6,6,6,6,6,0
1,1,e2,0,1,2e2,2e2,2e2,0,2e2,0
2,1,e12,0,2,2e1^e2,2e1^e2,2e1^e2,0,2e1^e2,0
3,1,e23,0,2,2e2^e3,2e2^e3,2e2^e3,0,2e2^e3,0
4,1,I,0,3,2e1^e2^e3,2e1^e2^e3,2e1^e2^e3,0,2e1^e2^e3,0
5,e1,1,1,0,3e1,3e1,0,3e1,3e1,0
6,e1,e2,1,1,e1^e2,e1^e2,0,0,0,0
7,e1,e12,1,2,e2,0,e2,0,e2,e2
8,e1,e23,1,2,e1^e2^e3,e1^e2^e3,0,0,0,0
9,e1,I,1,3,e2^e3,0,e2^e3,0,e2^e3,e2^e3


## Agreement And Branch Cases

The safest transfer rule is lower nonzero grade first. If `A` has nonzero grade and that grade is not larger than the grade of `B`, then left contraction, the Appendix B dot product, and the Hestenes inner product agree. That rule is why many derivations can be read across notational traditions without trouble. It is also why trouble can stay hidden: the examples in a text may all live in the agreement zone.

The next cell turns that rule into records. The examples include one agreement case, one grade-order branch, one scalar case, and one equal-grade scalar-product case. The artifact is intentionally explicit so tests can assert not just a value but the convention that produced it.


In [4]:
cases = {"vector_on_bivector": (e1, e12), "bivector_on_vector": (e12, e1), "scalar_on_vector": (alg.scalar(5), e1), "bivector_on_bivector": (e12, e12)}
case_rows = []
for name, (A, B) in cases.items():
    grade_A = next(iter(A.grades())) if A.grades() else 0
    grade_B = next(iter(B.grades())) if B.grades() else 0
    case_rows.append({"case":name,"grade_A":grade_A,"grade_B":grade_B,"left":repr(A.left_contract(B)),"right":repr(A.right_contract(B)),"dot":repr(dot_product(A,B)),"hestenes":repr(hestenes_inner(A,B))})
remember("agreement_cases_json", write_json_artifact([APPENDIX_KEY,"checks"], "agreement-and-branch-cases.json", case_rows))
pd.DataFrame(case_rows)


,case,grade_A,grade_B,left,right,dot,hestenes
0,vector_on_bivector,1,2,e2,0,e2,e2
1,bivector_on_vector,2,1,0,-e2,-e2,-e2
2,scalar_on_vector,0,1,5e1,0,5e1,0
3,bivector_on_bivector,2,2,-1,-1,-1,-1


## Grade Behavior

Grade behavior is the implementation test that catches most wrong inner products. Left contraction is only allowed to produce grade `s-r` from a grade `r` input and a grade `s` input, and it should be zero if that target grade is negative. Right contraction is the mirror. The dot product has grade `abs(r-s)` by construction. Hestenes has the same grade except when a scalar argument forces zero.

The grid below runs over every basis blade in a three-dimensional Euclidean algebra. It does not prove a full algebra implementation correct, but it makes grade mistakes visible and gives a compact reference artifact for convention-sensitive formulas.


In [5]:
blades = basis_blades(alg)
grade_rows = []
for left_name, A in blades.items():
    for right_name, B in blades.items():
        grade_rows.append({
            "left":left_name, "right":right_name,
            "grade_A":next(iter(A.grades())) if A.grades() else 0,
            "grade_B":next(iter(B.grades())) if B.grades() else 0,
            "left_result_grades":sorted(A.left_contract(B).grades()),
            "right_result_grades":sorted(A.right_contract(B).grades()),
            "dot_result_grades":sorted(dot_product(A,B).grades()),
            "hestenes_result_grades":sorted(hestenes_inner(A,B).grades()),
        })
remember("grade_behavior_csv", write_csv_artifact([APPENDIX_KEY,"tables"], "grade-behavior-grid.csv", grade_rows))
pd.DataFrame(grade_rows).head(16)


,left,right,grade_A,grade_B,left_result_grades,right_result_grades,dot_result_grades,hestenes_result_grades
0,1,1,0,0,[0],[0],[0],[]
1,1,e1,0,1,[1],[],[1],[]
2,1,e2,0,1,[1],[],[1],[]
3,1,e3,0,1,[1],[],[1],[]
4,1,e1^e2,0,2,[2],[],[2],[]
5,1,e1^e3,0,2,[2],[],[2],[]
6,1,e2^e3,0,2,[2],[],[2],[]
7,1,e1^e2^e3,0,3,[3],[],[3],[]
8,e1,1,1,0,[],[1],[1],[]
9,e1,e1,1,1,[0],[0],[0],[0]


## Recursive Vector-On-Blade Check

Appendix B spends serious effort relating explicit and implicit contraction definitions. The most reusable computational result is the alternating expansion for contracting a vector into a blade. In code, this is a good independent oracle because it computes the same value by a different route: expand over the blade factors, remove one factor at a time, and alternate signs.

The example uses a non-axis-aligned vector and a coordinate plane. The direct contraction and the recursive expansion must agree. If a sign convention in basis multiplication or contraction is wrong, this check usually fails immediately.


In [6]:
v = 2 * e1 - 3 * e2 + 5 * e3
plane = e1.wedge(e2)
direct = v.left_contract(plane)
recursive = vector_contract_blade_recursive(v, [e1, e2])
recursive_payload = {"vector":repr(v),"blade":repr(plane),"direct":repr(direct),"recursive":repr(recursive),"matches":direct.almost_equal(recursive)}
assert recursive_payload["matches"]
remember("recursive_contraction_json", write_json_artifact([APPENDIX_KEY,"checks"], "recursive-vector-on-blade.json", recursive_payload))
recursive_payload


{'vector': '2e1 - 3e2 + 5e3',
 'blade': 'e1^e2',
 'direct': '3e1 + 2e2',
 'recursive': '3e1 + 2e2',
 'matches': True}

## Projection, Norms, And Degeneracy

Projection is a sensitive convention test because it combines contraction, inverse, and geometric product. For a vector `x` and a non-null blade `B`, `(x left-contract B) B^{-1}` is the component of `x` inside the blade, while `(x wedge B) B^{-1}` is the rejected component. The two should add back to the original vector, and the rejected component should be orthogonal to the blade.

A degenerate metric adds a different warning. Contractions can still be evaluated deterministically, but inverse-based formulas may fail because a null blade has no stable inverse. The policy should be explicit: product definitions remain available where they are defined, while inverse-based operations assert non-nullness.


In [7]:
x_vec = 2 * e1 + 3 * e2 + 4 * e3
proj = projection_onto_blade(x_vec, plane)
rej = rejection_from_blade(x_vec, plane)
projection_payload = {"x":repr(x_vec),"blade":repr(plane),"projection":repr(proj),"rejection":repr(rej),"recomposes":(proj+rej).almost_equal(x_vec),"rejection_contracts_to_zero":rej.left_contract(plane).almost_equal(alg.scalar(0)),"projection_square":inner_scalar(proj,proj),"rejection_square":inner_scalar(rej,rej)}
assert projection_payload["recomposes"] and projection_payload["rejection_contracts_to_zero"]
remember("projection_norm_json", write_json_artifact([APPENDIX_KEY,"checks"], "projection-norm-check.json", projection_payload))

deg_alg = Algebra([1, 1, 0], names=["e1", "e2", "z"])
d1, d2, dz = deg_alg.basis(); deg_plane = d1.wedge(dz)
inverse_error = None
try: deg_plane.inverse_blade()
except Exception as exc: inverse_error = type(exc).__name__
degenerate_payload = {"metric":metric_signature([1,1,0]),"z_square":inner_scalar(dz,dz),"z_left_contract_plane":repr(dz.left_contract(deg_plane)),"plane_square":blade_square(deg_plane),"inverse_error":inverse_error}
assert inverse_error == "ZeroDivisionError"
remember("degenerate_contraction_json", write_json_artifact([APPENDIX_KEY,"checks"], "degenerate-contraction.json", degenerate_payload))
projection_payload | {"degenerate_inverse_error": inverse_error}


{'x': '2e1 + 3e2 + 4e3',
 'blade': 'e1^e2',
 'projection': '2e1 + 3e2',
 'rejection': '4e3',
 'recomposes': True,
 'rejection_contracts_to_zero': True,
 'projection_square': 13.0,
 'rejection_square': 16.0,
 'degenerate_inverse_error': 'ZeroDivisionError'}

## Implementation Tests, Pitfalls, And Takeaways

The final test suite encodes the convention contract. It checks that the Appendix B dot product equals left contraction in the lower-grade-first region, equals right contraction when the left grade is higher, and that the Hestenes variant zeros scalar cases. These tests are intentionally small but broad over basis blades.

The main pitfall is to read a dot without asking which dot it is. In one text it may mean scalar product for equal grades; in another it may mean Hestenes inner product; in this book the preferred metric product is usually contraction. The second pitfall is to simplify away zero results. A zero left contraction can mean that a containment relationship failed, not that the computation was uninteresting. Name the convention in code. Test scalar cases. Test both grade orders. Use lower-grade-first formulas when moving between traditions. Keep projection tests nearby, because they combine products in ways that expose sign and grade mistakes.


In [8]:
failures = []
for A_name, A in blades.items():
    for B_name, B in blades.items():
        grade_A = next(iter(A.grades())) if A.grades() else 0
        grade_B = next(iter(B.grades())) if B.grades() else 0
        expected_dot = A.left_contract(B) if grade_A <= grade_B else A.right_contract(B)
        if not dot_product(A,B).almost_equal(expected_dot): failures.append((A_name,B_name,"dot"))
        expected_hestenes = alg.scalar(0) if (grade_A == 0 or grade_B == 0) else expected_dot
        if not hestenes_inner(A,B).almost_equal(expected_hestenes): failures.append((A_name,B_name,"hestenes"))
test_payload = {"failure_count":len(failures),"failures":failures[:10],"tested_pairs":len(blades)**2}
assert not failures
remember("implementation_tests_json", write_json_artifact([APPENDIX_KEY,"checks"], "inner-product-implementation-tests.json", test_payload))
test_payload


{'failure_count': 0, 'failures': [], 'tested_pairs': 64}

In [9]:
assert not failures
assert projection_payload["recomposes"] and projection_payload["rejection_contracts_to_zero"]
assert recursive_payload["matches"]
sanity_payload = {"appendix":"B", "artifact_count_before_sanity":len(artifact_paths), "artifacts":artifact_paths}
sanity_path = remember("final_sanity", write_json_artifact([APPENDIX_KEY,"checks"], "final-sanity.json", sanity_payload))
missing = [str(path) for path in map(Path, artifact_paths.values()) if not Path(path).exists()]
assert not missing, missing
print(json.dumps({"appendix":"B", "artifact_count":len(artifact_paths), "sanity":str(sanity_path)}, indent=2))


{
  "appendix": "B",
  "artifact_count": 10,
  "sanity": "D:\\Geometry\\Geometric-Algebra-for-Computer-Science\\artifacts\\appendices\\appendix-b\\checks\\final-sanity.json"
}
